# Real-time customer segmentation tutorial

This Colab notebook was generated from the FeatureMesh docs tutorial.

Work top to bottom: first install FeatureMesh plus local Redis, then create clients, then load data and run the tutorial.

1. **Install infrastructure** — Python packages, start local services, smoke-test them.
2. **Create FeatureMesh clients** — Jupyter magic, `ServingClient`, and serving executors.
3. **Set up tutorial data**, then follow the tutorial sections in order.


## 1. Install infrastructure

Install the FeatureMesh client, start Redis in this Colab VM, and verify a write/read round trip before creating clients.


In [ ]:
%pip install -q "featuremesh[serving]" pandas redis


In [ ]:
!apt-get -qq update
!apt-get install -y redis-server
!redis-server --daemonize yes


In [ ]:
import time

time.sleep(1)

# Redis: verify connection and scalar/list round trips.
import redis
r = redis.Redis(host="127.0.0.1", port=6379, decode_responses=True)
print("Redis ping:", r.ping())
r.set("smoke_test", "hello", ex=60)
print("Redis read back:", r.get("smoke_test"))
r.delete("smoke_test")

r.rpush("smoke_list", "a", "b")
print("Redis list:", r.lrange("smoke_list", 0, -1))
r.delete("smoke_list")
print("Redis OK")


## 2. Create FeatureMesh clients

Load the Jupyter magic, create a local `ServingClient`, and wire the serving executors.


In [ ]:
%load_ext featuremesh


In [ ]:
from IPython.display import display
from featuremesh import RegistryDeployment, ServingClient, ServingDeployment, set_default
from featuremesh.helpers.sltest_executors import RedisSltExecutor, SqlSltExecutor
from redis import Redis

set_default("registry", RegistryDeployment.LOCAL)
set_default("serving", ServingDeployment.EMBEDDED)

serving_client = ServingClient()
set_default("client", serving_client)

SERVING_EXECUTORS = {}
SERVING_EXECUTORS["redis"] = RedisSltExecutor(
    Redis.from_url("redis://127.0.0.1:6379/0", decode_responses=True)
)

client = serving_client
print("FeatureMesh clients ready; serving executors: Redis")


This tutorial turns a customer promotion rule into a real-time feature backed by Redis, verifies live updates, and serves the result through a prepared statement.

Complete the [FeatureMesh homepage walkthrough](https://featuremesh.com/docs/tutorials/homepage/featuremesh) first if source features, `VARIANT()`, or serving refreshes are new. You need Redis and a serving client for the steps below.


## Set up the tutorial data

Seed three Redis customer hashes, then reset the tutorial namespace:


In [3]:
_result = SERVING_EXECUTORS['redis'].statement("""DEL tutorial:segment:1 tutorial:segment:2 tutorial:segment:3
HSET tutorial:segment:1 days_since_order 45 lifetime_value_cents 15000
HSET tutorial:segment:2 days_since_order 10 lifetime_value_cents 8000
HSET tutorial:segment:3 days_since_order 60 lifetime_value_cents 5000
""")
if _result.errors:
    raise RuntimeError(_result.errors)


In [4]:
%%featureql --client serving_client --hide-dataframe

DROP FEATURES IF EXISTS IN FM.TUTORIALS.REALTIME_SEGMENTATION UP TO LEVEL 9;


*** Warning(s) (1) ***

  1. [DROP-FEATURES-EXPRESSION-EMPTY] No features found in expression: *(FEATURES IF EXISTS IN FM.TUTORIALS.REALTIME_SEGMENTATION UP TO LEVEL 9) (acknowledge with ACK-UBI4)


## The serving data

Each customer has a Redis hash with two precomputed values:

- `days_since_order` measures recency.
- `lifetime_value_cents` avoids floating-point currency comparisons.

The promotion rule targets customers whose last order was at least 30 days ago and whose lifetime value is at least $100.

## Define the serving model

The model declares a `CUSTOMERS` entity and maps `CUSTOMER_ID` to a Redis hash key. `EXTERNAL_REDIS()` reads each field, and `SHOW_PROMO` combines the typed values into the business rule:


In [5]:
%%featureql --client serving_client

CREATE OR REPLACE FEATURES IN FM.TUTORIALS.REALTIME_SEGMENTATION AS
SELECT
    CUSTOMERS := ENTITY(),
    CUSTOMER_ID := INPUT(BIGINT#CUSTOMERS),
    REDIS_SOURCE := SOURCE_REDIS(
        'redis://127.0.0.1:6379'
        WITH (timeout='500ms')
    ),
    REDIS_KEY := 'tutorial:segment:' || UNSAFE_CAST(CUSTOMER_ID AS VARCHAR),
    DAYS_SINCE_ORDER := CAST(
        EXTERNAL_REDIS(KEY REDIS_KEY FIELD 'days_since_order' FROM REDIS_SOURCE)
        AS BIGINT
    ),
    LIFETIME_VALUE_CENTS := CAST(
        EXTERNAL_REDIS(KEY REDIS_KEY FIELD 'lifetime_value_cents' FROM REDIS_SOURCE)
        AS BIGINT
    ),
    SHOW_PROMO := DAYS_SINCE_ORDER >= 30 AND LIFETIME_VALUE_CENTS >= 10000
;


,FEATURE_NAME,STATUS,MESSAGE
0,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMERS,CREATED,Feature created as not exists
1,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,CREATED,Feature created as not exists
2,FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_SOURCE,CREATED,Feature created as not exists
3,FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_KEY,CREATED,Feature created as not exists
4,FM.TUTORIALS.REALTIME_SEGMENTATION.DAYS_SINCE_...,CREATED,Feature created as not exists
5,FM.TUTORIALS.REALTIME_SEGMENTATION.LIFETIME_VA...,CREATED,Feature created as not exists
6,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO,CREATED,Feature created as not exists


Persisting the source updates the registry. A local serving process needs an explicit refresh before it can use the connection:


In [6]:
%%featureql --client serving_client

REFRESH FEATURES FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_SOURCE;


,FEATURE,KIND,STATUS,MESSAGE
0,FM.TUTORIALS.REALTIME_SEGMENTATION.REDIS_SOURCE,SOURCE_REDIS,REFRESHED,


## Which customers receive the promotion?

The first customer meets both conditions. Customer 2 is too recent and customer 3 has insufficient lifetime value:


In [7]:
%%featureql --client serving_client

SELECT
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID := BIND_VALUES(ARRAY[1, 2, 3]),
    FM.TUTORIALS.REALTIME_SEGMENTATION.DAYS_SINCE_ORDER,
    FM.TUTORIALS.REALTIME_SEGMENTATION.LIFETIME_VALUE_CENTS,
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
;


,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.DAYS_SINCE_ORDER,FM.TUTORIALS.REALTIME_SEGMENTATION.LIFETIME_VALUE_CENTS,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
0,1,45,15000,True
1,2,10,8000,False
2,3,60,5000,False


## What happens when Redis changes?

Reverse ETL updates customer 2 to 40 days since the last order and $200 in lifetime value. The next serving query reads the new hash values immediately:


In [8]:
_result = SERVING_EXECUTORS['redis'].statement("""HSET tutorial:segment:2 days_since_order 40 lifetime_value_cents 20000
""")
if _result.errors:
    raise RuntimeError(_result.errors)


In [9]:
%%featureql --client serving_client

SELECT
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID := BIND_VALUE(2),
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
;


,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
0,2,True


The source definition did not change, so no connection refresh was needed.

## Compile the rule for serving

`PREPARED_STATEMENT()` compiles the promotion rule and declares `CUSTOMER_ID` as its input table:


In [10]:
%%featureql --client serving_client

CREATE OR REPLACE FEATURE FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS AS
PREPARED_STATEMENT(
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
    USING FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID
);


,FEATURE_NAME,STATUS,MESSAGE
0,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS,CREATED,Feature created as not exists


Refresh the prepared statement after persisting it:


In [11]:
%%featureql --client serving_client

REFRESH FEATURES FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS;


,FEATURE,KIND,STATUS,MESSAGE
0,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS,PREPARED_STATEMENT,REFRESHED,


Call the prepared feature with two request payloads. Each payload supplies the rows for the generated input table; the responses stay separate:


In [12]:
import json

_prepared_id = 'FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS'
_prepared_calls = [
    "{\"input_table_1\": [[1], [2], [3]]}",
    "{\"input_table_1\": [[2]]}",
]
for _payload in _prepared_calls:
    _inputs = json.loads(_payload) if isinstance(_payload, str) else _payload
    _result = serving_client.execute_prepared_statement(_prepared_id, _inputs)
    if _result.errors:
        raise RuntimeError(_result.errors)
    display(_result.dataframe)


,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS
0,1,True
1,2,True
2,3,False


,FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS
0,2,True


## Full query vs prepared statement

An ad-hoc FeatureQL query recompiles and plans on every call. A prepared statement reuses the compiled plan and only binds inputs. The loops below use the same three-customer payload 100 times each:


In [ ]:
import time

_N = 100
_full_query = """
SELECT
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID,
    FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO
FOR
    FM.TUTORIALS.REALTIME_SEGMENTATION.CUSTOMER_ID := BIND_VALUES(ARRAY[1, 2, 3])
;
"""
_prepared_id = "FM.TUTORIALS.REALTIME_SEGMENTATION.SHOW_PROMO_PS"
_prepared_inputs = {"input_table_1": [[1], [2], [3]]}

_t0 = time.perf_counter()
for _ in range(_N):
    _result = serving_client.query(_full_query)
    if _result.errors:
        raise RuntimeError(_result.errors)
_full_seconds = time.perf_counter() - _t0

_t0 = time.perf_counter()
for _ in range(_N):
    _result = serving_client.execute_prepared_statement(_prepared_id, _prepared_inputs)
    if _result.errors:
        raise RuntimeError(_result.errors)
_prepared_seconds = time.perf_counter() - _t0

print(
    f"Full FeatureQL query × {_N}: {_full_seconds:.3f}s "
    f"({_full_seconds / _N * 1000:.2f} ms/call)"
)
print(
    f"Prepared statement × {_N}: {_prepared_seconds:.3f}s "
    f"({_prepared_seconds / _N * 1000:.2f} ms/call)"
)
print(f"Speedup: {_full_seconds / _prepared_seconds:.1f}×")


## What's next

- [Redis data sources](https://featuremesh.com/docs/featuremesh/realtime_serving/external_redis) — key, hash, and table access patterns
- [Prepared statements](https://featuremesh.com/docs/featuremesh/realtime_serving/prepared_statements) — input grouping and serving payloads
- [Multi-source queries](https://featuremesh.com/docs/featuremesh/realtime_serving/multi_source_queries) — combine Redis with relational data


## Clean up

Drop tutorial features and fixture data so you can re-run the notebook from a clean state.


In [14]:
%%featureql --client serving_client --hide-dataframe

DROP FEATURES IF EXISTS IN FM.TUTORIALS.REALTIME_SEGMENTATION UP TO LEVEL 9;


In [15]:
_result = SERVING_EXECUTORS['redis'].statement("""DEL tutorial:segment:1 tutorial:segment:2 tutorial:segment:3
""")
if _result.errors:
    raise RuntimeError(_result.errors)


---

Source tutorial: [/docs/tutorials/serving/realtime](https://featuremesh.com/docs/tutorials/serving/realtime)
